# Pedestrian Data (Historical 2024-2025)

In [186]:
import pandas as pd
raw_ped = pd.read_csv('merged_pedestrian_counts.csv')
sensor_location = pd.read_csv('raw/pedestrian-counting-system-sensor-locations.csv')

In [187]:
raw_ped.head()

,ID,Location_ID,Sensing_Date,HourDay,Direction_1,Direction_2,Total_of_Directions,Sensor_Name,Location
0,672220240621,67,2024-06-21,22,175,307,482,FLDegS_T,"-37.81688755, 144.96562569"
1,1091020240724,109,2024-07-24,10,34,165,199,LatWill_T,"-37.81193681, 144.95621105"
2,2420240919,2,2024-09-19,4,1,2,3,Bou283_T,"-37.81380668, 144.96516718"
3,492220241018,49,2024-10-18,22,140,124,264,Eli501_T,"-37.80730068, 144.95956055"
4,651120240731,65,2024-07-31,11,703,384,1087,SwaCs_T,"-37.81569416, 144.9668064"


In [188]:
sensor_location.head()

,Location_ID,Sensor_Description,Sensor_Name,Installation_Date,Note,Location_Type,Status,Direction_1,Direction_2,Latitude,Longitude,Location
0,24,Spencer St-Collins St (North),Col620_T,2013-09-02,NaN,Outdoor,A,East,West,-37.818880,144.954492,"-37.81887963, 144.95449198"
1,25,Melbourne Convention Exhibition Centre,MCEC_T,2013-08-28,NaN,Outdoor,A,East,West,-37.824018,144.956044,"-37.82401776, 144.95604426"
2,36,Queen St (West),Que85_T,2015-01-20,"Pushbox Upgrade, 03/08/2023",Outdoor,A,North,South,-37.816525,144.961211,"-37.81652527, 144.96121062"
3,41,Flinders La-Swanston St (West),Swa31,2017-06-29,NaN,Outdoor,A,North,South,-37.816686,144.966897,"-37.81668634, 144.96689733"
4,44,Tin Alley-Swanston St (West),UM3_T,2015-04-15,"Pushbox Upgrade, 30/06/2023",Outdoor,A,North,South,-37.796987,144.964413,"-37.79698741, 144.96441306"


In [189]:
print(sensor_location.shape)
sensor_location.nunique()

(132, 12)


Location_ID           132
Sensor_Description    131
Sensor_Name           130
Installation_Date      78
Note                   20
Location_Type           2
Status                  1
Direction_1             3
Direction_2             3
Latitude              132
Longitude             132
Location              132
dtype: int64

In [190]:
print(raw_ped['Sensing_Date'].min())
print(raw_ped['Sensing_Date'].max())

ped_df = raw_ped[(raw_ped['Sensing_Date']>='2024-04-01')&(raw_ped['Sensing_Date']<'2025-04-01')]
ped_df = ped_df[['Location_ID', 
                           'Sensing_Date', 
                           'HourDay',
                           'Total_of_Directions'
                           ]]

print(ped_df['Sensing_Date'].min())
print(ped_df['Sensing_Date'].max())


2024-01-01
2025-04-23
2024-04-01
2025-03-31


In [191]:
def map_period(hour):
    if 0 <= hour <= 5:
        return '1. Late Night (12am-6am)'
    elif 6 <= hour <= 11:
        return '2. Morning (6am-12pm)'
    elif 12 <= hour <= 17:
        return '3. Afternoon (12pm-6pm)'
    else:
        return '4. Night (6pm-12am)'

ped_df['period_of_time'] = ped_df['HourDay'].apply(map_period)

ped_df.groupby('period_of_time').agg({'HourDay': ['min', 'max', pd.Series.nunique]})

HourDay            
                             min max nunique
period_of_time                              
1. Late Night (12am-6am)       0   5       6
2. Morning (6am-12pm)          6  11       6
3. Afternoon (12pm-6pm)       12  17       6
4. Night (6pm-12am)           18  23       6

In [192]:
ped_df = ped_df.groupby(['Location_ID', 'Sensing_Date', 'period_of_time'])['Total_of_Directions'].agg(
    total_pedestrian_count='sum',
    hours_covered='count' 
).reset_index()

ped_df['avg_hourly_pedestrian_count'] = (
    ped_df['total_pedestrian_count'] / ped_df['hours_covered']
)
ped_df.head()


,Location_ID,Sensing_Date,period_of_time,total_pedestrian_count,hours_covered,avg_hourly_pedestrian_count
0,1,2024-04-01,1. Late Night (12am-6am),327,6,54.500000
1,1,2024-04-01,2. Morning (6am-12pm),3326,6,554.333333
2,1,2024-04-01,3. Afternoon (12pm-6pm),4197,2,2098.500000
3,1,2024-04-01,4. Night (6pm-12am),48,1,48.000000
4,1,2024-04-02,1. Late Night (12am-6am),70,6,11.666667


In [193]:
ped_df = pd.merge(ped_df,
                  sensor_location[['Location_ID', 
                                   'Sensor_Description', 
                                   'Sensor_Name', 
                                   'Latitude',
                                   'Longitude'
                                   ]], 
                  on='Location_ID', how='left')
ped_df.columns = ped_df.columns.str.replace(r'[^\w]', '_', regex=True).str.lower()
# add time period
ped_df.head()

,location_id,sensing_date,period_of_time,total_pedestrian_count,hours_covered,avg_hourly_pedestrian_count,sensor_description,sensor_name,latitude,longitude
0,1,2024-04-01,1. Late Night (12am-6am),327,6,54.500000,Bourke Street Mall (North),Bou292_T,-37.813494,144.965153
1,1,2024-04-01,2. Morning (6am-12pm),3326,6,554.333333,Bourke Street Mall (North),Bou292_T,-37.813494,144.965153
2,1,2024-04-01,3. Afternoon (12pm-6pm),4197,2,2098.500000,Bourke Street Mall (North),Bou292_T,-37.813494,144.965153
3,1,2024-04-01,4. Night (6pm-12am),48,1,48.000000,Bourke Street Mall (North),Bou292_T,-37.813494,144.965153
4,1,2024-04-02,1. Late Night (12am-6am),70,6,11.666667,Bourke Street Mall (North),Bou292_T,-37.813494,144.965153


In [194]:
print(ped_df.shape)
ped_df.nunique()

(127291, 10)


location_id                       96
sensing_date                     365
period_of_time                     4
total_pedestrian_count         12089
hours_covered                      6
avg_hourly_pedestrian_count    16372
sensor_description                95
sensor_name                       94
latitude                          96
longitude                         96
dtype: int64

In [195]:
# Prepare export data
# ped_df.to_csv('pedestrian.csv', index=False)

# Testing: Pedestrian Data (API)

In [196]:
import urllib.request

url = 'https://discover.data.vic.gov.au/api/3/action/datastore_search?resource_id=3b4c9b89-c976-49e7-8358-075fe3699297'  
fileobj = urllib.request.urlopen(url)
print(fileobj.read())

b'{"help": "https://discover.data.vic.gov.au/api/3/action/help_show?name=datastore_search", "success": true, "result": {"include_total": true, "limit": 100, "records_format": "objects", "resource_id": "3b4c9b89-c976-49e7-8358-075fe3699297", "total_estimation_threshold": null, "records": [{"_id":1,"location_id":"65","sensing_datetime":"2024-06-12T13:57:00+00:00","sensing_date":"2024-06-12","sensing_time":"23:57","direction_1":"0","direction_2":"2","total_of_directions":"2"},{"_id":2,"location_id":"65","sensing_datetime":"2024-06-12T13:58:00+00:00","sensing_date":"2024-06-12","sensing_time":"23:58","direction_1":"1","direction_2":"0","total_of_directions":"1"},{"_id":3,"location_id":"65","sensing_datetime":"2024-06-12T14:02:00+00:00","sensing_date":"2024-06-13","sensing_time":"00:02","direction_1":"1","direction_2":"0","total_of_directions":"1"},{"_id":4,"location_id":"65","sensing_datetime":"2024-06-12T14:03:00+00:00","sensing_date":"2024-06-13","sensing_time":"00:03","direction_1":"1",

In [197]:
import urllib.request
import json 

url = 'https://data.melbourne.vic.gov.au/api/explore/v2.1/catalog/datasets/pedestrian-counting-system-past-hour-counts-per-minute/records?limit=20'

fileobj = urllib.request.urlopen(url)
response = fileobj.read()

# Convert bytes to string by decoding
response_str = response.decode('utf-8')

# Parse JSON
data = json.loads(response_str)

In [198]:
data

{'total_count': 37608,
 'results': [{'location_id': 3,
   'sensing_datetime': '2025-04-23T13:56:00+00:00',
   'sensing_date': '2025-04-23',
   'sensing_time': '23:56',
   'direction_1': 8,
   'direction_2': 5,
   'total_of_directions': 13},
  {'location_id': 3,
   'sensing_datetime': '2025-04-23T14:00:00+00:00',
   'sensing_date': '2025-04-24',
   'sensing_time': '00:00',
   'direction_1': 2,
   'direction_2': 10,
   'total_of_directions': 12},
  {'location_id': 3,
   'sensing_datetime': '2025-04-23T14:03:00+00:00',
   'sensing_date': '2025-04-24',
   'sensing_time': '00:03',
   'direction_1': 0,
   'direction_2': 6,
   'total_of_directions': 6},
  {'location_id': 3,
   'sensing_datetime': '2025-04-23T14:04:00+00:00',
   'sensing_date': '2025-04-24',
   'sensing_time': '00:04',
   'direction_1': 1,
   'direction_2': 3,
   'total_of_directions': 4},
  {'location_id': 3,
   'sensing_datetime': '2025-04-23T14:05:00+00:00',
   'sensing_date': '2025-04-24',
   'sensing_time': '00:05',
   'd

In [199]:
import pandas as pd
records = data['results']
df = pd.DataFrame(records)



# Lighting Data

In [200]:
raw_light = pd.read_csv('raw/street-lights-with-emitted-lux-level-council-owned-lights-only.csv')
raw_light.head()

,geo_point_2d,geo_shape,prop_id,name,addresspt1,xorg,ext_id,asset_clas,label,asset_type,...,northing,str_id,addresspt,asset_subt,xsource,profile,xdate,xdrawing,mcc_id,roadseg_id
0,"-37.81373899955726, 144.9421289998664","{""coordinates"": [144.9421289998664, -37.813738...",0,NaN,0.0,ESG,210,NaN,6.549,NaN,...,0.0,0,0,NaN,NaN,NaN,20140916,NaN,0,0
1,"-37.81374400011677, 144.94213900036905","{""coordinates"": [144.94213900036905, -37.81374...",0,NaN,0.0,ESG,214,NaN,6.158,NaN,...,0.0,0,0,NaN,NaN,NaN,20140916,NaN,0,0
2,"-37.813802999768036, 144.94222000027733","{""coordinates"": [144.94222000027733, -37.81380...",0,NaN,0.0,ESG,251,NaN,54.448,NaN,...,0.0,0,0,NaN,NaN,NaN,20140916,NaN,0,0
3,"-37.81380999995285, 144.9422400005824","{""coordinates"": [144.9422400005824, -37.813809...",0,NaN,0.0,ESG,259,NaN,98.143,NaN,...,0.0,0,0,NaN,NaN,NaN,20140916,NaN,0,0
4,"-37.81351900001735, 144.94179300024945","{""coordinates"": [144.94179300024945, -37.81351...",0,NaN,0.0,ESG,77,NaN,18.280,NaN,...,0.0,0,0,NaN,NaN,NaN,20140916,NaN,0,0


In [201]:
# lighting data last updated at 20140916
raw_light['xdate'].drop_duplicates()

0    20140916
Name: xdate, dtype: int64

In [202]:
import json
raw_light['geo_type'] = raw_light['geo_shape'].apply(lambda x: json.loads(x)['type'])
raw_light.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106037 entries, 0 to 106036
Data columns (total 22 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   geo_point_2d  106037 non-null  object 
 1   geo_shape     106037 non-null  object 
 2   prop_id       106037 non-null  int64  
 3   name          0 non-null       float64
 4   addresspt1    106037 non-null  float64
 5   xorg          106037 non-null  object 
 6   ext_id        106037 non-null  int64  
 7   asset_clas    0 non-null       float64
 8   label         106037 non-null  float64
 9   asset_type    0 non-null       float64
 10  easting       106037 non-null  float64
 11  northing      106037 non-null  float64
 12  str_id        106037 non-null  int64  
 13  addresspt     106037 non-null  int64  
 14  asset_subt    0 non-null       float64
 15  xsource       0 non-null       float64
 16  profile       0 non-null       float64
 17  xdate         106037 non-null  int64  
 18  xdra

In [203]:
# Columns with only 1 value
raw_light[[col for col in raw_light.columns if raw_light[col].nunique() == 1]].drop_duplicates()

,prop_id,addresspt1,xorg,easting,northing,str_id,addresspt,xdate,mcc_id,roadseg_id,geo_type
0,0,0.0,ESG,0.0,0.0,0,0,20140916,0,0,Point


In [204]:
light_df = raw_light[[col for col in raw_light.columns if raw_light[col].nunique() > 1]].copy()
light_df['latitude'] = light_df['geo_point_2d'].str.split(', ').str[0].astype(float)
light_df['longitude'] = light_df['geo_point_2d'].str.split(', ').str[1].astype(float)

light_df.head()

,geo_point_2d,geo_shape,ext_id,label,latitude,longitude
0,"-37.81373899955726, 144.9421289998664","{""coordinates"": [144.9421289998664, -37.813738...",210,6.549,-37.813739,144.942129
1,"-37.81374400011677, 144.94213900036905","{""coordinates"": [144.94213900036905, -37.81374...",214,6.158,-37.813744,144.942139
2,"-37.813802999768036, 144.94222000027733","{""coordinates"": [144.94222000027733, -37.81380...",251,54.448,-37.813803,144.942220
3,"-37.81380999995285, 144.9422400005824","{""coordinates"": [144.9422400005824, -37.813809...",259,98.143,-37.813810,144.942240
4,"-37.81351900001735, 144.94179300024945","{""coordinates"": [144.94179300024945, -37.81351...",77,18.280,-37.813519,144.941793


In [205]:
light_df['geo_shape'].value_counts()

geo_shape
{"coordinates": [144.9702759997199, -37.80109000004725], "type": "Point"}      1017
{"coordinates": [144.9888309998181, -37.81247299958573], "type": "Point"}       582
{"coordinates": [144.98933399955305, -37.81318699976632], "type": "Point"}      430
{"coordinates": [144.97094700016362, -37.81849700013838], "type": "Point"}      261
{"coordinates": [144.92446900052795, -37.79896899953943], "type": "Point"}      153
                                                                               ... 
{"coordinates": [144.93848600051092, -37.777256999506704], "type": "Point"}       1
{"coordinates": [144.938872999602, -37.77807599974197], "type": "Point"}          1
{"coordinates": [144.93850699942126, -37.77726099968011], "type": "Point"}        1
{"coordinates": [144.93888099969053, -37.7780800003798], "type": "Point"}         1
{"coordinates": [144.92438300037776, -37.797155999966655], "type": "Point"}       1
Name: count, Length: 97405, dtype: int64

In [206]:
light_df[light_df['geo_shape']=='{"coordinates": [144.9702759997199, -37.80109000004725], "type": "Point"}']

,geo_point_2d,geo_shape,ext_id,label,latitude,longitude
518,"-37.80109000004725, 144.9702759997199","{""coordinates"": [144.9702759997199, -37.801090...",14705,1.564,-37.80109,144.970276
519,"-37.80109000004725, 144.9702759997199","{""coordinates"": [144.9702759997199, -37.801090...",14729,1.564,-37.80109,144.970276
525,"-37.80109000004725, 144.9702759997199","{""coordinates"": [144.9702759997199, -37.801090...",14918,1.564,-37.80109,144.970276
526,"-37.80109000004725, 144.9702759997199","{""coordinates"": [144.9702759997199, -37.801090...",14934,1.564,-37.80109,144.970276
527,"-37.80109000004725, 144.9702759997199","{""coordinates"": [144.9702759997199, -37.801090...",15240,1.564,-37.80109,144.970276
...,...,...,...,...,...,...
96510,"-37.80109000004725, 144.9702759997199","{""coordinates"": [144.9702759997199, -37.801090...",15182,1.564,-37.80109,144.970276
96527,"-37.80109000004725, 144.9702759997199","{""coordinates"": [144.9702759997199, -37.801090...",15557,1.564,-37.80109,144.970276
96528,"-37.80109000004725, 144.9702759997199","{""coordinates"": [144.9702759997199, -37.801090...",15567,1.564,-37.80109,144.970276
96529,"-37.80109000004725, 144.9702759997199","{""coordinates"": [144.9702759997199, -37.801090...",15584,1.564,-37.80109,144.970276


In [207]:
# drop duplicates based on geo_shape, keeping only the record with latest ext_id
light_df = light_df.sort_values('ext_id', ascending=False)
light_df = light_df.drop_duplicates(subset=['geo_shape'])


In [208]:
light_df.head()

,geo_point_2d,geo_shape,ext_id,label,latitude,longitude
68412,"-37.81910299993481, 144.94717399954112","{""coordinates"": [144.94717399954112, -37.81910...",106037,18.280,-37.819103,144.947174
56442,"-37.81910499968281, 144.94717700002684","{""coordinates"": [144.94717700002684, -37.81910...",106029,12.512,-37.819105,144.947177
71876,"-37.81910800031837, 144.94717900034982","{""coordinates"": [144.94717900034982, -37.81910...",106028,12.414,-37.819108,144.947179
75245,"-37.81911000004645, 144.9471819997004","{""coordinates"": [144.9471819997004, -37.819110...",106027,12.317,-37.819110,144.947182
51779,"-37.81911200038785, 144.9471840004734","{""coordinates"": [144.9471840004734, -37.819112...",106026,12.414,-37.819112,144.947184


In [209]:
light_df.nunique()

geo_point_2d    97405
geo_shape       97405
ext_id          97405
label            1016
latitude        97401
longitude       97398
dtype: int64

In [210]:
light_df[light_df['geo_shape']=='{"coordinates": [144.9702759997199, -37.80109000004725], "type": "Point"}']

,geo_point_2d,geo_shape,ext_id,label,latitude,longitude
15244,"-37.80109000004725, 144.9702759997199","{""coordinates"": [144.9702759997199, -37.801090...",15628,1.564,-37.80109,144.970276


In [211]:
light_df = light_df[['latitude', 'longitude', 'ext_id', 'label']].rename(columns={'label': 'emitted_lux_level'})

# Geoencoding: Postcode with australian_postcodes

In [212]:
postcodes = pd.read_csv('raw/australian_postcodes.csv')
postcodes.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18526 entries, 0 to 18525
Data columns (total 41 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                18526 non-null  int64  
 1   postcode          18526 non-null  int64  
 2   locality          18526 non-null  object 
 3   state             18526 non-null  object 
 4   long              18526 non-null  float64
 5   lat               18526 non-null  float64
 6   dc                17929 non-null  object 
 7   type              18403 non-null  object 
 8   status            18500 non-null  object 
 9   sa3               18365 non-null  float64
 10  sa3name           18365 non-null  object 
 11  sa4               18365 non-null  float64
 12  sa4name           18365 non-null  object 
 13  region            18526 non-null  object 
 14  Lat_precise       18526 non-null  float64
 15  Long_precise      18526 non-null  float64
 16  SA1_CODE_2021     17943 non-null  float6

In [213]:
postcodes[postcodes['state']=='VIC'].head(3).T

,6194,6195,6196
id,4746,4747,4748
postcode,3000,3001,3002
locality,MELBOURNE,MELBOURNE,EAST MELBOURNE
state,VIC,VIC,VIC
long,144.982585,144.982585,144.982585
lat,-37.814437,-37.814437,-37.814437
dc,CITY DELIVERY CENTRE,CITY MAIL PROCESSING CENTRE,CITY DELIVERY CENTRE
type,Delivery Area,Post Office Boxes,Delivery Area
status,Updated 17-Mar-2024 AUSPOST,Updated 17-Mar-2024 AUSPOST,Updated 17-Mar-2024 AUSPOST
sa3,20604.0,20605.0,20604.0


In [214]:
vic_postcodes = postcodes[postcodes['state']=='VIC'][['postcode', 'locality','state', 'Lat_precise', 'Long_precise']].drop_duplicates()
vic_postcodes = vic_postcodes.rename(columns={'Lat_precise': 'lat',
                              'Long_precise': 'long'})
vic_postcodes.head()

,postcode,locality,state,lat,long
6194,3000,MELBOURNE,VIC,-37.815207,144.963937
6195,3001,MELBOURNE,VIC,-37.813628,144.963058
6196,3002,EAST MELBOURNE,VIC,-37.816144,144.980459
6197,3003,WEST MELBOURNE,VIC,-37.811450,144.925397
6198,3004,MELBOURNE,VIC,-37.830158,144.980459


In [215]:
vic_postcodes.nunique()

postcode     750
locality    3343
state          1
lat         3337
long        3163
dtype: int64

In [216]:
vic_postcodes[['postcode', 'locality']].value_counts()

postcode  locality        
3000      MELBOURNE           1
3660      WHITEHEADS CREEK    1
          CAVEAT              1
          DROPMORE            1
          DYSART              1
                             ..
3364      COGHILLS CREEK      1
          GLENDONALD          1
          GLENDONNELL         1
          JOYCES CREEK        1
9999      NORTH POLE          1
Name: count, Length: 3538, dtype: int64

In [217]:
vic_postcodes[vic_postcodes['locality']=='DOCKLANDS']

,postcode,locality,state,lat,long
6204,3008,DOCKLANDS,VIC,-37.817065,144.941912
18497,8012,DOCKLANDS,VIC,-37.819000,144.947000


In [218]:
from scipy.spatial import cKDTree

tree = cKDTree(vic_postcodes[['lat', 'long']].values)

distances, indices = tree.query(ped_df[['latitude', 'longitude']], k=1)
print(ped_df.shape)
ped_df['postcode'] = vic_postcodes.iloc[indices]['postcode'].values
ped_df['locality'] = vic_postcodes.iloc[indices]['locality'].values

print(ped_df.shape)
ped_df.to_csv('pedestrian.csv', index=False)
ped_df.head()


(127291, 10)
(127291, 12)


,location_id,sensing_date,period_of_time,total_pedestrian_count,hours_covered,avg_hourly_pedestrian_count,sensor_description,sensor_name,latitude,longitude,postcode,locality
0,1,2024-04-01,1. Late Night (12am-6am),327,6,54.500000,Bourke Street Mall (North),Bou292_T,-37.813494,144.965153,8120,MELBOURNE
1,1,2024-04-01,2. Morning (6am-12pm),3326,6,554.333333,Bourke Street Mall (North),Bou292_T,-37.813494,144.965153,8120,MELBOURNE
2,1,2024-04-01,3. Afternoon (12pm-6pm),4197,2,2098.500000,Bourke Street Mall (North),Bou292_T,-37.813494,144.965153,8120,MELBOURNE
3,1,2024-04-01,4. Night (6pm-12am),48,1,48.000000,Bourke Street Mall (North),Bou292_T,-37.813494,144.965153,8120,MELBOURNE
4,1,2024-04-02,1. Late Night (12am-6am),70,6,11.666667,Bourke Street Mall (North),Bou292_T,-37.813494,144.965153,8120,MELBOURNE


In [219]:
ped_df[['postcode','locality']].value_counts()

postcode  locality              
8009      FLINDERS LANE             26538
8007      COLLINS STREET WEST       12618
9999      NORTH POLE                12497
8006      ABECKETT STREET           12037
8011      LITTLE LONSDALE STREET    10130
8120      MELBOURNE                  7181
3002      EAST MELBOURNE             7136
3008      DOCKLANDS                  6882
3051      HOTHAM HILL                5836
8012      DOCKLANDS                  5761
3053      CARLTON SOUTH              5588
3052      MELBOURNE UNIVERSITY       5251
3005      WORLD TRADE CENTRE         4254
3031      KENSINGTON                 2920
3053      CARLTON                    1460
3006      SOUTHBANK                  1119
3051      NORTH MELBOURNE              83
Name: count, dtype: int64

In [220]:
distances, indices = tree.query(light_df[['latitude', 'longitude']], k=1)
light_df['postcode'] = vic_postcodes.iloc[indices]['postcode'].values
light_df['locality'] = vic_postcodes.iloc[indices]['locality'].values
light_df.to_csv('street_light.csv', index=False)
light_df.head()


,latitude,longitude,ext_id,emitted_lux_level,postcode,locality
68412,-37.819103,144.947174,106037,18.280,8012,DOCKLANDS
56442,-37.819105,144.947177,106029,12.512,8012,DOCKLANDS
71876,-37.819108,144.947179,106028,12.414,8012,DOCKLANDS
75245,-37.819110,144.947182,106027,12.317,8012,DOCKLANDS
51779,-37.819112,144.947184,106026,12.414,8012,DOCKLANDS


In [221]:
light_df[['postcode','locality']].value_counts()

postcode  locality              
3002      EAST MELBOURNE            13161
3008      DOCKLANDS                 13084
8012      DOCKLANDS                 10946
3032      TRAVANCORE                 9034
8009      FLINDERS LANE              8489
3052      PARKVILLE                  6445
3053      CARLTON SOUTH              5659
3031      KENSINGTON                 5648
3053      CARLTON                    4299
3054      PRINCES HILL               3654
3005      WORLD TRADE CENTRE         3488
9999      NORTH POLE                 2828
8120      MELBOURNE                  2283
8002      EAST MELBOURNE             2236
8011      LITTLE LONSDALE STREET     2232
3004      ST KILDA ROAD CENTRAL      1502
3205      SOUTH MELBOURNE             636
8007      COLLINS STREET WEST         526
3121      RICHMOND NORTH              497
8008      ST KILDA ROAD CENTRAL       434
3006      SOUTH WHARF                 295
3141      DOMAIN ROAD PO               29
Name: count, dtype: int64

In [222]:
vic_postcodes[['postcode', 'locality']].to_csv('vic_postcode.csv', index=False)